# Stock Analysis Notebook

Exploratory analysis and experimentation for the Stock Market Prediction project.

> Educational purposes only — not financial advice.

In [ ]:
import sys
sys.path.append('..')

from utils.data_loader import fetch_stock_data
from utils.preprocessing import clean_data, scale_close_prices, create_sequences, train_test_split_sequence
from utils.indicators import calculate_all_indicators
from utils.ml_models import train_linear_regression, train_random_forest, build_lstm_model, train_lstm_model
from utils.evaluation import evaluate_predictions

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# 1. Fetch data
symbol = 'AAPL'
raw_df = fetch_stock_data(symbol, '2019-01-01', '2024-12-31')
df = clean_data(raw_df)
df.tail()

In [ ]:
# 2. Plot closing price
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'])
plt.title(f'{symbol} Closing Price')
plt.xlabel('Date')
plt.ylabel('Price')
plt.show()

In [ ]:
# 3. Technical indicators
df_ind = calculate_all_indicators(df)
df_ind[['Date', 'Close', 'MA20', 'MA50', 'RSI', 'MACD']].tail()

In [ ]:
# 4. Prepare sequences for LSTM
window_size = 60
scaled_data, scaler = scale_close_prices(df)
X, y = create_sequences(scaled_data, window_size)
X_train, X_test, y_train, y_test = train_test_split_sequence(X, y)
X_train.shape, X_test.shape

In [ ]:
# 5. Train LSTM
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
history = train_lstm_model(model, X_train, y_train, epochs=15)

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('LSTM Training Loss')
plt.show()

In [ ]:
# 6. Evaluate
pred_scaled = model.predict(X_test)
pred = scaler.inverse_transform(pred_scaled)
actual = scaler.inverse_transform(y_test.reshape(-1, 1))

metrics = evaluate_predictions(actual, pred)
print(metrics)

plt.figure(figsize=(12, 5))
plt.plot(actual, label='Actual')
plt.plot(pred, label='Predicted')
plt.legend()
plt.title(f'{symbol} — Actual vs Predicted (LSTM)')
plt.show()